In [1]:
import ale_py
import torch
import numpy as np
import gymnasium as gym
import random
from model import DeepQNetwork
from data import ReplayBuffer, Transition

gym.register_envs(ale_py)

In [2]:
num_frames = 4
img_height = 84
img_width = 84

def make_env(env_id: str):
    env = gym.make(env_id, frameskip=1)
    env = gym.wrappers.AtariPreprocessing(
        env=env,
        screen_size=img_height,
        grayscale_obs=True,
        frame_skip=4,
        terminal_on_life_loss=False,
        scale_obs=False
    )
    env = gym.wrappers.FrameStackObservation(env, stack_size=num_frames)
    env = gym.wrappers.RecordEpisodeStatistics(env)    
    
    return env

env = make_env("ALE/SpaceInvaders-v5")
env.reset();

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


In [3]:
predictor_model = DeepQNetwork(
    img_height=img_height, 
    img_width=img_width, 
    action_space=6, 
    num_frames=4,
).to('cuda')

# used to provide stable q-values for bellman equation.
target_model =  DeepQNetwork(
    img_height=img_height, 
    img_width=img_width, 
    action_space=6, 
    num_frames=4,
).to('cuda')

target_model.load_state_dict(predictor_model.state_dict())
target_model.eval()

Linear Dim 64
Linear Dim 64


DeepQNetwork(
  (conv_layers): Sequential(
    (conv2d): Conv2d(4, 8, kernel_size=(8, 8), stride=(2, 2))
    (silu): GELU(approximate='none')
    (conv2d2): Conv2d(8, 16, kernel_size=(4, 4), stride=(2, 2))
    (silu2): GELU(approximate='none')
    (conv2d3): Conv2d(16, 32, kernel_size=(4, 4), stride=(2, 2))
    (silu3): GELU(approximate='none')
    (conv2d4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2))
    (silu4): GELU(approximate='none')
    (conv2d5): Conv2d(64, 64, kernel_size=(2, 2), stride=(2, 2))
    (silu5): GELU(approximate='none')
  )
  (ff_layers): Sequential(
    (linear1): Linear(in_features=64, out_features=512, bias=True)
    (gelu): GELU(approximate='none')
    (linear2): Linear(in_features=512, out_features=64, bias=True)
    (gelu2): GELU(approximate='none')
    (linear3): Linear(in_features=64, out_features=512, bias=True)
    (gelu3): GELU(approximate='none')
    (action_linear): Linear(in_features=512, out_features=6, bias=True)
  )
)

In [4]:
state, info = env.reset()
# print(state)
state_t = torch.from_numpy(state).to(dtype=torch.float32, device='cuda') / 255
print(state.shape)
print(state_t.shape)
print(predictor_model(state_t.unsqueeze(0)))

(4, 84, 84)
torch.Size([4, 84, 84])
tensor([[ 0.0208, -0.0093,  0.0578, -0.0578,  0.0445,  0.0301]],
       device='cuda:0', grad_fn=<AddmmBackward0>)


In [5]:
def train_step(
    replay_buffer: ReplayBuffer, 
    batch_size: int,
    optimizer,
    predictor_model,
    target_model,
    gamma,
    loss_func
) -> float:
    if len(replay_buffer) < batch_size:
        return

    states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

    predicted_q_values = predictor_model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q_values = target_model(next_states).max(dim=1).values

    # Bellman equation.
    target_q_values = rewards + gamma * next_q_values * (1 - dones)

    loss = loss_func(predicted_q_values, target_q_values)

    optimizer.zero_grad() # clear grads

    loss.backward() # calc grads.

    optimizer.step() # update model

    return loss.item()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from pathlib import Path

episodes = 10000
step_counter = 0
terminated = False
replay_buffer = ReplayBuffer(capacity=5000)
learning_starts = 500 # keep warmup proportional to the smaller buffer
reward_ma_window = 50
plot_every = 20
plot_path = Path('/home/ubuntu/repos/deep-rl/models/atari/runs/notebook_training_progress.png')
plot_path.parent.mkdir(parents=True, exist_ok=True)

from torch.optim import AdamW

gamma = 0.99
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay_steps=100000
batch_size = 32
target_update_freq = 2000
learning_rate = 1e-5

optimizer = AdamW(params=predictor_model.parameters(), lr=learning_rate)

"""huber loss"""
def loss_func(pred, target, delta=1.0):
    diff = target - pred
    abs_diff = torch.abs(diff)
    quadratic = 0.5 * (diff) ** 2
    linear = delta * ((abs_diff)  -  0.5 * delta)
    loss = torch.where(abs_diff <= delta, quadratic, linear)
    return loss.mean()


def moving_average(values, window):
    averages = []
    for idx in range(len(values)):
        start = max(0, idx - window + 1)
        window_values = values[start:idx + 1]
        averages.append(sum(window_values) / len(window_values))
    return averages

# loss_func = torch.nn.MSELoss()

episode_rewards = []
episode_avg_losses = []

for episode in range(episodes):
    state, info = env.reset()
    terminated = False
    truncated = False
    episode_reward = 0.0
    episode_losses = []

    while not terminated and not truncated:
        state_t = torch.as_tensor(state, dtype=torch.float32, device='cuda') / 255.0
        progress = min(step_counter / epsilon_decay_steps, 1.0)
        epsilon = epsilon_start + progress * (epsilon_end - epsilon_start)

        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                predicted_q_values = predictor_model(state_t.unsqueeze(0))
            action = predicted_q_values.argmax(dim=1).item()

        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

        done = terminated or truncated
        replay_buffer.push(
            Transition(
                torch.from_numpy(state).to(device='cuda', dtype=torch.float32),
                action,
                reward,
                torch.from_numpy(next_state).to(device='cuda', dtype=torch.float32),
                done,
            )
        )

        if step_counter >= learning_starts:
            loss_value = train_step(
                replay_buffer=replay_buffer, 
                batch_size=batch_size,
                optimizer=optimizer,
                predictor_model=predictor_model,
                target_model=target_model,
                gamma=gamma,
                loss_func=loss_func
            )
            if loss_value is not None:
                episode_losses.append(loss_value)

        state = next_state
        step_counter += 1

        if step_counter % target_update_freq == 0:
            target_model.load_state_dict(predictor_model.state_dict())

    episode_rewards.append(episode_reward)
    if episode_losses:
        episode_avg_losses.append(sum(episode_losses) / len(episode_losses))
    else:
        episode_avg_losses.append(float('nan'))

    if episode % plot_every == 0 and episode_rewards:
        fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
        reward_ax, loss_ax = axes

        reward_ma = moving_average(episode_rewards, reward_ma_window)
        reward_ax.plot(episode_rewards, alpha=0.35, color='tab:blue', linewidth=0.8, label='Reward')
        reward_ax.plot(reward_ma, color='tab:orange', linewidth=2.0, label=f'{reward_ma_window}-ep MA')
        reward_ax.set_ylabel('Reward')
        reward_ax.legend(loc='upper left')
        reward_ax.set_title(
            f'Episode {episode} · reward {episode_rewards[-1]:.0f} · '
            f'max {max(episode_rewards):.0f} · ε {epsilon:.3f}'
        )

        loss_ax.plot(episode_avg_losses, alpha=0.8, color='tab:red', linewidth=0.8)
        loss_ax.set_xlabel('Episode')
        loss_ax.set_ylabel('Avg Loss')
        loss_ax.set_title('Average training loss per episode')

        fig.tight_layout()
        fig.savefig(plot_path)
        plt.close(fig)

        print(f'episode={episode} reward={episode_reward:.2f} epsilon={epsilon:.3f} saved={plot_path}')

episode=0 reward=120.00 epsilon=0.995 saved=/home/ubuntu/repos/deep-rl/models/atari/runs/notebook_training_progress.png
episode=20 reward=310.00 epsilon=0.906 saved=/home/ubuntu/repos/deep-rl/models/atari/runs/notebook_training_progress.png
episode=40 reward=150.00 epsilon=0.812 saved=/home/ubuntu/repos/deep-rl/models/atari/runs/notebook_training_progress.png
episode=60 reward=30.00 epsilon=0.717 saved=/home/ubuntu/repos/deep-rl/models/atari/runs/notebook_training_progress.png
episode=80 reward=180.00 epsilon=0.608 saved=/home/ubuntu/repos/deep-rl/models/atari/runs/notebook_training_progress.png
